# 05 — SST-anomaly composites

Per-ENSO-year fields (Jun(0)–May(+1)) are composited by phase. Composite = mean over
phase years − mean over all ENSO years {ENSO_YEARS}, after removing a linear trend across years if
`cfg.COMPOSITE_DETREND`. Significance: resampling null (random draws of the same number of years,
`cfg.N_BOOT`), two-sided; maps stippled where significant after Benjamini–Hochberg FDR
(`ALPHA_FDR`, Wilks 2016). Regional curves: filled markers = pointwise p < 0.05.
With n≈3–4 strong El Niños the null is coarse — treat individual-event lines as part of the result.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ausenso as cfg
warnings.filterwarnings("ignore", category=RuntimeWarning)
cfg.ensure_dirs()
table = cfg.load_enso_table()
groups = cfg.phase_years(table)
masks = xr.open_dataset(cfg.F_MASKS)
names = masks.attrs["region_names"].split(";")
comps = ["El Niño", "Strong El Niño", "La Niña"]
{k: [table.label[y] for y in groups[k]] for k in comps}

In [ ]:
ssta = xr.open_dataset(cfg.F_SST_MONTHLY).T_anom
ym = cfg.to_enso_year_month(ssta)            # (enso_year, emonth, lat, lon)
seas = cfg.to_seasons(ym, "mean")            # (enso_year, season, lat, lon)
res_map = {k: cfg.composite_bootstrap(seas, groups[k]) for k in comps}

In [ ]:
fig = cfg.plot_composite_maps(res_map, "SST anomaly (°C)", vmax=1.0, masks=masks,
                              title=f"SST anomaly composites (baseline {cfg.BASELINE})")
fig.savefig(cfg.FIG / "05_sst_maps.png", dpi=200, bbox_inches="tight")

In [ ]:
# El Niño minus La Niña difference map (simple, no resampling)
d = (res_map["El Niño"].comp - res_map["La Niña"].comp)
fig, axs = cfg.map_axes(1, 4, figsize=(14, 3.2), ocean=masks.ocean)
import cartopy.crs as ccrs
for j, s in enumerate(d.season.values):
    pm = axs[0, j].pcolormesh(d.lon, d.lat, d.sel(season=s).where(masks.ocean), vmin=-1.5, vmax=1.5,
                              cmap="RdBu_r", transform=ccrs.PlateCarree(), shading="auto")
    axs[0, j].set_title(f"EN − LN {s}", fontsize=8)
fig.colorbar(pm, ax=axs, shrink=0.7, label="°C")

## Regional shelf-mean lag evolution

In [ ]:
reg = cfg.regional_means(ym, masks)          # (region, enso_year, emonth)
res_reg = {k: cfg.composite_bootstrap(reg, groups[k]) for k in comps}
indiv = (cfg.detrend_years(reg) if cfg.COMPOSITE_DETREND else reg)
indiv = indiv - indiv.mean("enso_year")
indiv.attrs["years"] = groups["Strong El Niño"]
fig = cfg.plot_regional_lags(res_reg, "SST anomaly (°C)", names, individual=indiv)
fig.savefig(cfg.FIG / "05_sst_regional.png", dpi=200, bbox_inches="tight")

In [ ]:
# Shelf vs slope contrast, all regions pooled
pooled = xr.concat([cfg.box_mean(ym, mask=masks.shelf), cfg.box_mean(ym, mask=masks.slope)], "band").assign_coords(band=["shelf", "slope"])
tab = pd.DataFrame({k: cfg.composite_bootstrap(pooled, groups[k]).comp.to_pandas().stack() for k in comps})
tab.round(2)

In [ ]:
out = xr.concat([r.drop_vars(["null_lo", "null_hi"], errors="ignore").expand_dims(composite=[k])
                 for k, r in res_map.items()], "composite")
out.to_netcdf(cfg.OUT / "composites_sst_maps.nc")
xr.concat([r.expand_dims(composite=[k]) for k, r in res_reg.items()], "composite").to_netcdf(cfg.OUT / "composites_sst_regional.nc")